# Chapter 3 — Cubic Equations of State: Scientific Figures

**Figures:**
1. van der Waals isotherm and Maxwell construction
2. SRK vs PR alpha functions and comparison
3. Volume translation effect on liquid density
4. Generalized cubic P-V isotherms at sub- and supercritical T

In [1]:
import importlib, subprocess, sys
try:
    from neqsim_dev_setup import neqsim_init, neqsim_classes
    ns = neqsim_init(recompile=False); ns = neqsim_classes(ns)
    NEQSIM_MODE = "devtools"
except Exception:
    try: import neqsim
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "neqsim"])
    from neqsim import jneqsim
    NEQSIM_MODE = "pip"
print(f"NeqSim mode: {NEQSIM_MODE}")

NeqSim project root: C:\Users\ESOL\Documents\GitHub\neqsim


NeqSim mode: pip


In [2]:
import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PALETTE = ["#2171b5", "#e6550d", "#31a354", "#756bb1", "#e7298a", "#66a61e"]
BLUE, ORANGE, GREEN, PURPLE, PINK, LIME = PALETTE
plt.rcParams.update({"font.family": "serif", "font.size": 9, "axes.labelsize": 10, "axes.titlesize": 10, "legend.fontsize": 8, "xtick.labelsize": 8, "ytick.labelsize": 8, "xtick.direction": "in", "ytick.direction": "in", "axes.linewidth": 0.6, "lines.linewidth": 1.2, "grid.linewidth": 0.3, "grid.alpha": 0.4, "savefig.dpi": 300, "figure.dpi": 150})
FIGURES_DIR = Path("../figures"); FIGURES_DIR.mkdir(exist_ok=True)
def save(fig, name): fig.savefig(str(FIGURES_DIR / name), dpi=300, bbox_inches="tight", pad_inches=0.05); plt.close(fig); print(f"Saved: {name}")

R = 8.314  # J/(mol K)

## Figure 3.1 — van der Waals Isotherms with Maxwell Construction

In [3]:
# van der Waals: P = RT/(V-b) - a/V^2
# Generic parameters for CO2-like molecule: Tc=304.2K, Pc=73.8 bar
Tc, Pc = 304.2, 73.8e5  # K, Pa
a_vdw = 27 * R**2 * Tc**2 / (64 * Pc)
b_vdw = R * Tc / (8 * Pc)

V = np.linspace(1.2 * b_vdw, 10 * b_vdw, 1000)

fig, ax = plt.subplots(figsize=(3.5, 3.0))

for Tr, col, ls in [(0.85, BLUE, "-"), (0.95, ORANGE, "-"), (1.0, "black", "-"), (1.1, GREEN, "--")]:
    T_val = Tr * Tc
    P_val = R * T_val / (V - b_vdw) - a_vdw / V**2
    P_bar = P_val / 1e5
    ax.plot(V * 1e6, P_bar, color=col, linestyle=ls, linewidth=1.2,
            label=f"$T_r = {Tr}$")

ax.set_xlabel(r"Molar volume $V$ (cm$^3$/mol)")
ax.set_ylabel("Pressure (bar)")
ax.set_title("van der Waals Isotherms")
ax.set_ylim(-20, 200)
ax.set_xlim(40, 600)
ax.axhline(y=0, color="grey", linestyle="-", alpha=0.3, linewidth=0.5)
ax.legend(frameon=True, fontsize=7)
ax.grid(True, alpha=0.3)

# Mark critical point
Vc = 3 * b_vdw
ax.plot(Vc * 1e6, Pc / 1e5, 'ko', markersize=6, zorder=5)
ax.annotate("Critical point", xy=(Vc * 1e6, Pc / 1e5),
            xytext=(Vc * 1e6 + 100, Pc / 1e5 + 30), fontsize=7,
            arrowprops=dict(arrowstyle="->", color="black", lw=0.5))

fig.tight_layout()
save(fig, "fig_ch03_01_vdw_isotherms.png")

Saved: fig_ch03_01_vdw_isotherms.png


## Figure 3.2 — Alpha Functions: SRK vs PR vs Twu

In [4]:
Tr = np.linspace(0.4, 2.0, 500)
omega = 0.344  # water

# SRK alpha
m_srk = 0.48 + 1.574 * omega - 0.176 * omega**2
alpha_srk = (1 + m_srk * (1 - np.sqrt(Tr)))**2

# PR alpha
m_pr = 0.37464 + 1.54226 * omega - 0.26992 * omega**2
alpha_pr = (1 + m_pr * (1 - np.sqrt(Tr)))**2

# Mathias-Copeman (3-param, illustrative)
c1, c2, c3 = 1.2, -0.5, 0.3
alpha_mc = np.where(Tr <= 1.0,
    (1 + c1*(1-np.sqrt(Tr)) + c2*(1-np.sqrt(Tr))**2 + c3*(1-np.sqrt(Tr))**3)**2,
    (1 + c1*(1-np.sqrt(Tr)))**2)

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(Tr, alpha_srk, color=BLUE, linewidth=1.5, label="SRK")
ax.plot(Tr, alpha_pr, color=ORANGE, linewidth=1.5, linestyle="--", label="PR")
ax.plot(Tr, alpha_mc, color=GREEN, linewidth=1.5, linestyle="-.", label="Mathias\u2013Copeman")
ax.axvline(x=1.0, color="grey", linestyle=":", alpha=0.5)
ax.annotate(r"$T_r = 1$ (critical)", xy=(1.0, 0.3), fontsize=7, color="grey")
ax.set_xlabel(r"Reduced temperature $T_r$")
ax.set_ylabel(r"$\alpha(T_r)$")
ax.set_title(r"Alpha functions ($\omega = 0.344$, water)")
ax.legend(frameon=True); ax.grid(True, alpha=0.3)
ax.set_ylim(0, 6)
fig.tight_layout()
save(fig, "fig_ch03_02_alpha_functions.png")

Saved: fig_ch03_02_alpha_functions.png


## Figure 3.3 — SRK vs PR Liquid Density (NeqSim)

Compares SRK and PR predictions for methane liquid density along the
saturation curve.

In [5]:
if NEQSIM_MODE == "pip":
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
else:
    from neqsim import jneqsim
    SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
    SystemPrEos = jneqsim.thermo.system.SystemPrEos
    ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations

temps_K = np.arange(95, 186, 5)  # methane: Tb=111.6K, Tc=190.6K
data_srk = {"T": [], "rho": []}
data_pr = {"T": [], "rho": []}

for T_K in temps_K:
    for name, cls, data in [("SRK", SystemSrkEos, data_srk), ("PR", SystemPrEos, data_pr)]:
        try:
            f = cls(float(T_K), 1.0)
            f.addComponent("methane", 1.0)
            f.setMixingRule("classic")
            ops = ThermodynamicOperations(f)
            ops.bubblePointPressureFlash(False)
            f.initProperties()
            rho = float(f.getPhase("oil").getDensity("kg/m3"))
            data["T"].append(float(T_K))
            data["rho"].append(rho)
        except Exception:
            pass

# NIST data for methane saturated liquid
nist_T = [100, 120, 140, 160, 180]
nist_rho = [422.4, 395.5, 361.8, 317.6, 249.7]

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(data_srk["T"], data_srk["rho"], color=BLUE, linewidth=1.5, label="SRK")
ax.plot(data_pr["T"], data_pr["rho"], color=ORANGE, linewidth=1.5, linestyle="--", label="PR")
ax.scatter(nist_T, nist_rho, color="black", s=25, zorder=5, label="NIST",
           edgecolors="black", linewidths=0.5)
ax.set_xlabel("Temperature (K)")
ax.set_ylabel("Liquid density (kg/m\u00b3)")
ax.set_title("Methane saturated liquid density")
ax.legend(frameon=True); ax.grid(True, alpha=0.3)
fig.tight_layout()
save(fig, "fig_ch03_03_srk_pr_methane_density.png")

Saved: fig_ch03_03_srk_pr_methane_density.png


## Figure 3.4 — Volume Translation Concept

In [6]:
# Illustration: shift PV isotherm by constant c
Tr_val = 0.85
T_val = Tr_val * Tc
V_range = np.linspace(1.3 * b_vdw, 8 * b_vdw, 500)
P_original = R * T_val / (V_range - b_vdw) - a_vdw / V_range**2

c_shift = 0.3 * b_vdw  # volume translation parameter
V_translated = V_range - c_shift

fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.plot(V_range * 1e6, P_original / 1e5, color=BLUE, linewidth=1.5, label="Original")
ax.plot(V_translated * 1e6, P_original / 1e5, color=ORANGE, linewidth=1.5,
        linestyle="--", label=f"Translated ($c = {c_shift*1e6:.0f}$ cm\u00b3/mol)")

ax.annotate("", xy=(100, 40), xytext=(115, 40),
            arrowprops=dict(arrowstyle="<->", color="red", lw=1.0))
ax.text(107, 43, r"$c$", fontsize=9, color="red", ha="center")

ax.set_xlabel(r"Molar volume (cm$^3$/mol)")
ax.set_ylabel("Pressure (bar)")
ax.set_title("P\u00e9neloux Volume Translation")
ax.legend(frameon=True, fontsize=7); ax.grid(True, alpha=0.3)
ax.set_ylim(-20, 150)
fig.tight_layout()
save(fig, "fig_ch03_04_volume_translation.png")

Saved: fig_ch03_04_volume_translation.png
